# Student Performance Analysis and Prediction System

**Minor Project | AI/ML with Python Training Program**  
**Prepared by:** Shivam A Pandya  
**Program:** B.Tech - AI Engineering, SRM Ramapuram

This notebook follows the project workflow: inspect and clean the data, perform exploratory analysis, plot academic patterns, train Linear Regression, and evaluate it on an unseen test partition.

**Dataset note:** This project uses selected fields from the UCI Student Performance Portuguese-course dataset (649 records). `studytime` is a weekly band and `absences` is an absence count; neither is converted into an invented exact-hours or attendance-percentage field. See `data/source_info.md` for attribution and full variable notes.

## 1. Import libraries and set project paths

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from model import prepare_data, make_eda, train_evaluate

DATA_PATH = PROJECT_ROOT / "data" / "student_performance_dataset.csv"
print("Project folder:", PROJECT_ROOT)
print("Dataset exists:", DATA_PATH.exists())

## 2. Load and inspect the dataset

In [ ]:
df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
display(df.head())
print("\nColumn types:")
display(df.dtypes.to_frame("data_type"))
print("\nMissing values:")
display(df.isna().sum().to_frame("missing_count"))

## 3. Data cleaning and feature engineering

The model uses `studytime`, `absences`, `G1` and `G2` to estimate `G3`. Two descriptive groups (`studytime_band` and `absence_group`) are derived for EDA. The complete source file was checked for exact duplicates: none were found. Repeated combinations in the five selected columns are not deleted because different students can share the same selected values.

In [ ]:
data, cleaning_summary = prepare_data()
print(json.dumps(cleaning_summary, indent=2))
display(data.head())

## 4. Descriptive statistics

In [ ]:
summary = data[["studytime", "absences", "G1", "G2", "G3"]].describe().T
summary["median"] = data[["studytime", "absences", "G1", "G2", "G3"]].median()
display(summary.round(3))
print(f"Mean G3: {data['G3'].mean():.3f}")
print(f"Median G3: {data['G3'].median():.3f}")
print(f"Standard deviation of G3: {data['G3'].std():.3f}")

## 5. Explore performance groups and relationships

In [ ]:
eda_summary = make_eda(data)
print("Correlation with G3:")
display(pd.Series(eda_summary["correlations_with_G3"], name="correlation").to_frame())
print("\nMean final grade by weekly study-time band:")
display(pd.read_csv(PROJECT_ROOT / "outputs" / "evaluation_results" / "studytime_group_summary.csv").round(3))
print("\nMean final grade by absence group:")
display(pd.read_csv(PROJECT_ROOT / "outputs" / "evaluation_results" / "absence_group_summary.csv").round(3))

## 6. Required visualizations

The script saves a bar chart, histogram, three scatter plots, a box plot and a correlation heatmap in `outputs/visualizations/`.

In [ ]:
from IPython.display import display, Image
plot_dir = PROJECT_ROOT / "outputs" / "visualizations"
plot_files = [
    "bar_mean_grade_by_studytime.png",
    "histogram_final_grade.png",
    "scatter_studytime_vs_final_grade.png",
    "scatter_absences_vs_final_grade.png",
    "scatter_g2_vs_final_grade.png",
    "boxplot_grade_by_studytime.png",
    "correlation_heatmap.png",
]
for filename in plot_files:
    print(f"Saved visualization: outputs/visualizations/{filename}")
    display(Image(filename=str(plot_dir / filename), width=650))

## 7. Train the Linear Regression model

An 80/20 train-test split is used. Imputation and scaling are inside a Scikit-learn pipeline, so those transformations are fitted only on training folds. A small grid search compares the intercept setting using 5-fold cross-validation.

In [ ]:
metrics = train_and_evaluate(data)
print(json.dumps(metrics, indent=2))

## 8. Model evaluation and predictions

In [ ]:
results_dir = PROJECT_ROOT / "outputs" / "evaluation_results"
predictions = pd.read_csv(results_dir / "test_predictions.csv")
print("Evaluation metrics:")
display(pd.Series({
    "MAE": metrics["test_MAE"],
    "RMSE": metrics["test_RMSE"],
    "R2": metrics["test_R2"],
    "5-fold CV MAE mean": metrics["cross_validation_MAE_mean"],
    "5-fold CV MAE standard deviation": metrics["cross_validation_MAE_std"],
    "5-fold CV R2 mean": metrics["cross_validation_R2_mean"],
}).to_frame("value").round(3))
print("\nSample actual and predicted G3 values:")
display(predictions.head(12).round(2))

## 9. Interpretation, limitations and conclusion

In this run, G2 has the strongest linear relationship with G3 (correlation approximately 0.919), followed by G1 (approximately 0.826). Study-time category has a smaller positive relationship (approximately 0.250), while absences show a weak negative relationship (approximately -0.091). On the held-out test set, MAE is approximately 0.748, RMSE is approximately 1.171 and R-squared is approximately 0.859.

These are associations from a dataset covering two Portuguese schools, not causal effects or universal findings. The strongest predictors are earlier period grades, so this model is appropriate only after G1 and G2 are available. `studytime` is a weekly category and `absences` is a count rather than attendance percentage. Predictions should be treated as estimates to support review, not as a decision about a student.